# Methodology {#sec-methodology}

*Describe what was done, what approach was followed, and how it was applied to the specific problem. Not a textbook chapter on research methodology. The reader does not need to be taught what deductive reasoning is, what qualitative versus quantitative means, or what "transferability, dependability, confirmability" stand for. Cover the philosophy of method in at most one short paragraph, then spend the rest of the chapter on what was actually done.*

A useful test: would a competent peer at another university know what to buy, how to wire it up, and how to crunch the numbers to repeat the measurement, after reading this chapter? If yes, the chapter is right-sized. If they would need to read another textbook first, you have written one yourself. Cut it.

Group activities by what they accomplished, not by what week they happened. This is a methodology chapter, not a project diary.


## Research design

*Name the type of study (experimental, comparative, case study, design science, simulation-based). Justify the choice against the research questions. The link should be obvious: the design is the cheapest way to credibly answer the RQs.*

Most engineering theses fall into one of three patterns:

- **Comparative experiment.** Build two or more variants of a system, run them under matched conditions, measure the difference. Typical when the RQ has the form "does X outperform Y?".
- **Design science.** Build an artefact that solves a stated problem, evaluate it against the requirements. Typical when the RQ has the form "how should X be designed to achieve Y?".
- **Simulation study.** Develop or use a model, vary inputs, characterise outputs. Typical when physical experiments are too expensive, slow or dangerous.

State which pattern the thesis follows in the opening sentence.


## Data collection

*Describe sensors, instruments, software, test rigs and procedures. Give part numbers and software versions. State sampling rates and resolutions. If interviews or surveys were used, describe the population, the sampling method and the instrument.*

A useful test: would a peer at another university know what to buy and how to wire it up to repeat your measurement? If not, add detail. If a procedure is long, summarise it here and put the step-by-step in the appendix.


## Data analysis

*Describe how the raw data was processed. Filtering, statistical tests, FEA solvers, fitting procedures: all belong here. State the metric used to compare conditions (rise time, ISE, RMSE, p-value with the chosen threshold).*

Example from a finite-element study. The simulation in this thesis used ANSYS Mechanical 18.2. The model contained three components, evaluated on a symmetrical plane to reduce the solve time. Static structural analysis was chosen because the loading was steady, and the solver used an unsymmetrical Newton-Raphson method to handle the non-linear contact problem [@Logan2012; @Stromberg2012].

Convergence was monitored across load steps; an example mesh-independence study is included in the appendix.


## Validity and reliability

*State what was done to ensure the results actually measure what they claim to measure (validity) and that re-running the procedure would yield the same numbers (reliability). Be self-critical; an honest acknowledgement of a weakness reads better than a glossed-over one.*

Examples of validity threats and the typical mitigation:

- **Model fidelity.** The CAD model omits manufacturing dilution at the bend radius, which would slightly reduce the spring stiffness. Mitigation: a sensitivity study on wall thickness, reported in the appendix.
- **Parameter uncertainty.** Friction coefficients were taken from a generic table rather than measured on the actual surface roughness. Mitigation: simulations were repeated at the upper and lower bounds of the tabulated range.
- **Reproducibility.** All scripts and CAD files are committed to the project repository at the tags listed in the README.


## Code listings

*Code goes in fenced blocks. To make a listing referenceable, give it an id, a class for the language and a caption. The same rules apply as for figures: reference the listing from the prose, and do not include a listing the reader is not told to look at.*

Example of a referenceable C listing showing the core PID update loop used on the embedded controller (@lst-c-pid).

```{#lst-c-pid .c lst-cap="PID controller update loop in C, suitable for an embedded controller."}
#include <stdio.h>

typedef struct {
    double kp, ki, kd;
    double integral;
    double prev_error;
} PIDController;

double pid_compute(PIDController *pid, double setpoint,
                   double measured, double dt) {
    double error = setpoint - measured;
    pid->integral += error * dt;
    double derivative = (error - pid->prev_error) / dt;
    pid->prev_error = error;

    return pid->kp * error
         + pid->ki * pid->integral
         + pid->kd * derivative;
}
```

Quarto can also execute and render code from notebook cells. The Python cell below is configured with `eval: false` so the listing renders without running on every build, which is the safer default for a thesis. Set `eval: true` once the simulation is final.


In [ ]:
#| label: lst-python-analysis
#| lst-cap: "PID step response simulation in Python."
#| eval: false

import numpy as np
import matplotlib.pyplot as plt

def simulate_pid(kp, ki, kd, setpoint=0.0, initial_angle=5.0,
                 dt=0.01, steps=500):
    """Simulate a PID-controlled inverted pendulum."""
    angle = initial_angle
    integral = 0.0
    prev_error = 0.0
    angles = []

    for _ in range(steps):
        error = setpoint - angle
        integral += error * dt
        derivative = (error - prev_error) / dt
        prev_error = error

        output = kp * error + ki * integral + kd * derivative
        angle += output * dt
        angles.append(angle)

    return np.array(angles)

time = np.arange(500) * 0.01
conservative = simulate_pid(kp=1.0, ki=0.1, kd=0.05)
aggressive   = simulate_pid(kp=3.0, ki=0.8, kd=0.2)

plt.plot(time, conservative, label="Conservative")
plt.plot(time, aggressive,   label="Aggressive")
plt.xlabel("Time [s]")
plt.ylabel("Angle [deg]")
plt.legend()
plt.show()

@lst-c-pid and @lst-python-analysis use the same control law, but the C version targets an MCU running at 200 Hz while the Python version runs offline for analysis and plotting.


## Executable code and the PDF {#sec-python-quarto}

Notebook code cells are not just for visual display; Quarto can **execute** them as part of the render and capture the output (text, tables, plots, derived equations) directly into the PDF. This is the literate-programming half of Quarto: prose, code, and the result of that code live in the same document.

### The pipeline

What happens when you run `quarto render`:

1. Quarto reads each `.ipynb` listed in `_quarto.yml`.
2. For every code cell with `eval: true`, Quarto sends the source to a Jupyter kernel (Python in this template; R via Knitr and Julia also work).
3. The kernel returns the captured output: standard-output text, generated images (PNG or PDF), DataFrame representations, LaTeX strings.
4. Pandoc converts the markdown plus captured output into LaTeX.
5. LuaLaTeX produces the PDF.

The PDF therefore reflects the *result of running the analysis*, not the source alone. If the analysis is non-deterministic (random seeds, network calls, hardware in the loop), pin the seeds and commit the data so future renders match.

### Cell directives

Each Python cell starts with one or more `#|` directives that control execution and rendering:

- `#| eval: true` runs the cell. `#| eval: false` skips it. The template default is `false` (set in `_quarto.yml`), so a half-finished cell never breaks the render.
- `#| echo: true` shows the source listing. `#| echo: false` hides it (the output still appears).
- `#| output: false` hides the captured output. Combined with `echo: true`, the cell renders as a listing only.
- `#| output: asis` pipes the cell's text output back into Pandoc as markdown. Essential for sympy-generated equations and pandas-generated tables.
- `#| label: fig-name` and `#| fig-cap: "Caption."` register the cell's plot as a referenceable figure.
- `#| label: tbl-name` and `#| tbl-cap: "Caption."` do the same for tables.
- `#| fig-width: 5` and `#| fig-height: 3` size the matplotlib figure in inches.

### Generating figures with matplotlib

The most common executable cell is a matplotlib plot. Quarto captures the figure and inserts it as a referenceable figure with caption and cross-link:

```python
#| label: fig-step-response
#| fig-cap: "Step response of the closed-loop system, comparing conservative and aggressive PID gains."
#| fig-width: 5
#| fig-height: 3
#| eval: false   # flip to true for the final render

import numpy as np
import matplotlib.pyplot as plt

t = np.linspace(0, 5, 500)
y_cons = 1 - np.exp(-2*t)
y_agg  = 1 - np.exp(-5*t) * np.cos(8*t)

plt.plot(t, y_cons, label="Conservative")
plt.plot(t, y_agg,  label="Aggressive")
plt.xlabel("Time [s]")
plt.ylabel("Output")
plt.legend()
plt.show()
```

With `eval: true`, the PDF contains the plot under an auto-numbered caption, and prose can cross-reference it as `@fig-step-response`.

For publication-quality plots, raise the resolution and prefer vector output:

```python
import matplotlib
matplotlib.rcParams["figure.dpi"]     = 200
matplotlib.rcParams["savefig.format"] = "pdf"
matplotlib.rcParams["font.size"]      = 9
```

PDF output stays sharp at any zoom level in the final document. A font size of 9 pt in the plot matches the surrounding body text once the figure is scaled to ~70 % of text width.

### Generating equations with sympy

Sympy can derive equations symbolically and emit the LaTeX. With `#| output: asis`, that LaTeX is passed straight to Pandoc and shows up in the PDF as a properly numbered display equation:

```python
#| output: asis
#| eval: false

import sympy as sp

t, omega, zeta, K = sp.symbols("t omega zeta K", positive=True)
s = sp.symbols("s")

# Closed-loop transfer function and its step response.
H_s = K / (s**2 + 2*zeta*omega*s + omega**2)
y_t = sp.inverse_laplace_transform(H_s / s, s, t)
y_t = sp.simplify(y_t)

print(r"$$")
print(sp.latex(y_t))
print(r"$$ {#eq-step-symbolic}")
```

The PDF then contains an equation labelled `eq-step-symbolic`, ready to reference from prose as `@eq-step-symbolic`. Change the damping ratio, the natural frequency, or the input shape, and the equation regenerates on the next render. Hand-copied equations drift; symbolically derived equations stay synchronised with the analysis.

Sympy is also useful for parameter substitution and partial simplification:

```python
#| output: asis
#| eval: false

import sympy as sp

E, I, L, F = sp.symbols("E I L F", positive=True)
delta_tip = F * L**3 / (3 * E * I)            # cantilever tip deflection

# Substitute concrete values for an aluminium I-beam, print the simplified
# numerical version next to the symbolic one.
vals = {E: 70e9, I: 8.5e-6, L: 1.2, F: 250}
print(r"$$")
print(sp.latex(sp.Eq(sp.Symbol(r"\delta_\text{tip}"), delta_tip)))
print(r"\quad\Rightarrow\quad")
print(sp.latex(sp.Eq(sp.Symbol(r"\delta_\text{tip}"),
                     sp.nsimplify(delta_tip.subs(vals), rational=False))))
print(r"$$ {#eq-cantilever-tip}")
```

### Generating tables with pandas

Markdown tables can also be computed:

```python
#| label: tbl-pid-metrics
#| tbl-cap: "PID metrics across ten runs."
#| output: asis
#| eval: false

import pandas as pd
df = pd.DataFrame({
    "Metric":         ["Rise time [s]", "Settling time [s]", "Overshoot [%]"],
    "Fixed-gain PID": [0.42, 1.85, 18.3],
    "Adaptive PID":   [0.31, 1.12,  8.7],
})
print(df.to_markdown(index=False))
```

`#| output: asis` feeds the printed markdown table back through Pandoc, which produces a proper LaTeX table with the caption attached and an entry in the List of Tables.

### Project-wide execution defaults

The `_quarto.yml` `execute:` block sets the defaults for every cell:

```yaml
execute:
  eval: false   # do not run cells unless explicitly enabled
  echo: true    # always show the source listing
```

`eval: false` is the safe default during drafting: a slow simulation, a half-finished analysis, or a missing data file does not break the render. Override per-cell with `#| eval: true` once the cell is ready.

### Reproducibility note

For the final-pass render, the analysis should run end-to-end on the examiner's machine. That means pinned random seeds, versioned input data, and a committed `requirements.txt` (or `environment.yml`) listing the Python packages and versions used. The pipeline above only reproduces what the cell can do today; what runs in two years depends on what is committed to the repository.


## Use of artificial intelligence

*Jönköping University requires a disclosure of how AI tools were used. Be specific; vague disclosures invite follow-up questions at the examination. Adapt the wording below to what was actually done.*

This thesis was conducted with the support of AI-based language models, specifically ChatGPT (OpenAI) and Claude (Anthropic). The tools were used in the following capacities:

- **Literature search and synthesis.** AI was used to identify relevant search terms and to summarise key findings from published articles. All sources were independently verified by the authors before inclusion in the report.
- **Writing assistance.** Drafts of selected sections were refined with AI support for grammar, clarity and academic tone. The intellectual content, argumentation and conclusions remain entirely the authors' own work.
- **Code development.** AI tools assisted in generating and debugging control algorithm implementations. All code was reviewed, tested and validated by the authors on the physical robot platform.
- **Mathematical verification.** Equation derivations and unit consistency checks were cross-checked using AI to reduce the risk of algebraic errors.

AI-generated output was never used verbatim without review. The authors take full responsibility for the accuracy, originality and integrity of the work presented in this report. The use of AI in this thesis complies with the guidelines set by Jönköping University regarding AI-assisted academic work.
